# Text Moderation (TF toxicity)

# Text Moderation — TF toxicity classifier (Reddit IRL + profanity/slang lexicons)

Blends the big Reddit IRL corpus (`the-reddit-irl-dataset-{comments,posts}.csv`),
the profanity lexicon (`profanity_en.csv`), Gen-Z slang
(`genz_slang_usage_2020_2025.csv`), and (optional) Jigsaw via HF. Feeds the
`app/moderation_engine.py` `/api/v1/moderation/text` endpoint with a
**multilingual + slang-aware** model that must stay schema-compatible with the
serving engine.

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'tensorflow_text'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}
from tf_utils import set_memory_growth
set_memory_growth()


In [ ]:
import pandas as pd
from buddy_data import profanity, genz_slang, reddit_irl_comments
prof = profanity()
slang = genz_slang()
irl = reddit_irl_comments()
print('profanity rows:', len(prof), '| slang:', len(slang), '| reddit IRL:', len(irl))

In [ ]:
# Simple subword/text-vectorizer over the combined corpus
import tensorflow as tf
texts = irl['body'].astype(str).tolist()[:200_000] if 'body' in irl else irl.iloc[:,0].astype(str).tolist()[:200_000]
vectorizer = tf.keras.layers.TextVectorization(max_tokens=60_000, output_sequence_length=256)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(texts).batch(512))
print('vocab size:', vectorizer.vocabulary_size())

In [ ]:
# Labels: toxicity prior from the profanity lexicon + slang intensity
# (self/train-bootstrapped). Jigsaw HF dataset is the gold external source.
import re
bad = set(prof['text'].astype(str).str.lower())
def tox(s):
    s = s.lower()
    return 1.0 if any(w in s for w in bad) else 0.0
y = [tox(t) for t in texts]

In [ ]:
from tf_utils import build_text_toxicity_model
m = build_text_toxicity_model(vocab_size=60_000, seq_len=256)
m.summary()

In [ ]:
import numpy as np
import tensorflow as tf
X = vectorizer(np.array(texts)).numpy()
m.fit(X, np.array(y), epochs=3, validation_split=0.1)

In [ ]:
from pathlib import Path
import tensorflow as tf
import tf2onnx
onnx_path = Path('../models/toxicity_classifier-1.0.0.onnx')
spec = [tf.TensorSpec((None,), tf.string, name='input_text')]
tf2onnx.convert.from_keras(m, signature_def={'serving_default': spec}, output_path=str(onnx_path))
print('exported', onnx_path)